# RPO

Rendezvous with the target in the Mars parking orbit, starting at the capture burn from
`arrival.ipynb`. Impulsive 3-DOF guidance on a J2 truth propagation:

1. Plane change at the node (optionally pre-biased against J2 nodal drift)
2. Hohmann to the target radius, then phasing entry
3. Closed-loop same-orbit phasing (re-planned each rev)
4. Phase exit, then HCW two-impulse terminal rendezvous

Runs a nominal (exact capture) and a dispersed case and saves both to `rpo-<mission>.mission`.
The original step-by-step plan lives in `rpo_legacy.ipynb`.

In [66]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [67]:
from pathlib import Path

import numpy as np
from cesium.constants import (
    DEG_TO_RAD,
    HOUR_TO_SEC,
    KM_TO_M,
    M_TO_KM,
    MIN_TO_SEC,
    RAD_TO_DEG,
    SEC_TO_MIN,
)
from cesium.guidance.hcw import RpoConfig, dv_breakdown, dv_breakdown_table, run_rpo
from cesium.math.time import jd_to_greg, jd_to_sec_j2000
from cesium.orbital.coes import COES
from cesium.plotting import plot_3d, plot_rsw, plot_separation
from cesium.world.constants import J2_MARS, MU_MARS, R_MARS
from mission_io import load_mission, save_mission
from numpy.linalg import norm

## Setup
Wired to `arrival.ipynb`: the timeline starts at the capture epoch, the target is the
arrival parking orbit, and the chaser starts from the post-capture MCI state. Two cases:

- **nominal**: exactly the post-capture state (coplanar with the target, so phasing only)
- **dispersed**: nominal + seeded capture-execution errors (±100 km a, e ≤ 0.01, ±3° i/RAAN/aop)

In [68]:
# From arrival.ipynb: capture epoch, post-capture state, approach hyperbola, target orbit (MCI)
MISSION_NAME = "birthday-christmas-earlier"
arrival = load_mission("arrival", MISSION_NAME)
EPOCH0_JD_TDB = float(arrival["epoch_jd"])
EPOCH0_TDB_S = jd_to_sec_j2000(EPOCH0_JD_TDB)
rv_capture_mci_m = arrival["rv_capture_mci_m"]
t_approach_s, rv_approach_mci_m = arrival["time_s"], arrival["rv_mci_m"]
DV_CAPTURE_M_S = float(arrival["dv_capture_m_s"])

# Target: the arrival parking orbit. Its phase at the capture epoch is a free choice.
TARGET_TA_DEG = 13
TARGET_0 = COES.from_array(arrival["target_coes"]).with_ta(TARGET_TA_DEG * DEG_TO_RAD)
TARGET_0.print("target at capture")

target at capture: a = 3696.000 km, e = 0.00000, i = 45.000 deg, raan = 15.000 deg, aop = 0.000 deg, ta = 13.000 deg


In [69]:
J2 = J2_MARS  # 0 for two-body truth

# Nominal chaser: exactly the post-capture state
CHASER_NOMINAL = COES.from_rv(rv_capture_mci_m, MU_MARS)

# Dispersed chaser: nominal + seeded capture-execution errors
SEED = 13
ANGLE_DISP_DEG = 3.0  # i, raan, aop
A_DISP_M = 100 * KM_TO_M
E_MAX = 0.01  # e >= 0, so draw from [0, E_MAX]
rng = np.random.default_rng(SEED)
rel_coes = np.array(
    [
        rng.uniform(-A_DISP_M, A_DISP_M),
        rng.uniform(0.0, E_MAX),
        *rng.uniform(-ANGLE_DISP_DEG, ANGLE_DISP_DEG, 3) * DEG_TO_RAD,
        0.0,  # ta: keep the arrival phase
    ]
)
CHASER_DISPERSED = COES.from_array(CHASER_NOMINAL.elements() + rel_coes)

CASES = {"nominal": CHASER_NOMINAL, "dispersed": CHASER_DISPERSED}
for name, chaser0 in CASES.items():
    chaser0.print(f"{name} chaser")
print(f"phase gap at capture (chaser - target aol): "
      f"{(CHASER_NOMINAL.aol - TARGET_0.aol) * RAD_TO_DEG % 360:.2f} deg")

nominal chaser: a = 3696.000 km, e = 0.00000, i = 45.000 deg, raan = 15.000 deg, aop = 0.000 deg, ta = 279.223 deg
dispersed chaser: a = 3768.960 km, e = 0.00855, i = 46.866 deg, raan = 13.569 deg, aop = -2.537 deg, ta = 279.223 deg
phase gap at capture (chaser - target aol): 266.22 deg


In [70]:
print(
    "RPO start (capture burn): {:04.0f}-{:02.0f}-{:02.0f} {:02.0f}:{:02.0f}:{:06.3f} TDB".format(
        *jd_to_greg(EPOCH0_JD_TDB)
    ),
    f"({EPOCH0_TDB_S:.3f} s past J2000); capture dv {DV_CAPTURE_M_S:.1f} m/s",
)

RPO start (capture burn): 2035-12-25 00:00:00.000 TDB (1135425600.000 s past J2000); capture dv 5524.4 m/s


## Burn timeline
One propagated truth for both spacecraft (`cesium.guidance.RpoTimeline`): every `coast` continues from the last state, every `burn` is applied to it, and each burn is planned from the current (propagated) COES. Times are seconds since `EPOCH0_TDB_S`; `tl.epochs_tdb_s()` and `tl.burn_table()` give absolute epochs.

- LOWER ORBITS DO RAAN FASTER

In [71]:
def raan_dot_j2(a, e, i):
    """Secular J2 nodal regression rate [rad/s]."""
    n = np.sqrt(MU_MARS / a**3)
    p = a * (1 - e**2)
    return -1.5 * n * J2 * (R_MARS / p) ** 2 * np.cos(i)

In [72]:
CFG = RpoConfig(
    mu=MU_MARS,
    j2=J2,
    r_cb=R_MARS,
    epoch0_tdb_s=EPOCH0_TDB_S,
    k_phase=20,
    hcw_thresh=20 * KM_TO_M,
    t_rendezvous=40 * MIN_TO_SEC,
    dock_thresh=50,  # m
)

def solve_case(name, chaser0):
    """Pass 1 measures the J2 RAAN drift during phasing; pass 2 pre-biases Burn 1. Keeps the cheaper."""
    print(f"===== {name}")
    tl_0, info_0 = run_rpo(chaser0, TARGET_0, CFG, verbose=True)

    # Analytic estimate: phasing orbit sits higher, so its node regresses more slowly
    drift_analytic = (
        raan_dot_j2(info_0["a_phase"], 0, info_0["i_target"])
        - raan_dot_j2(info_0["a_target"], 0, info_0["i_target"])
    ) * info_0["t_phase"]
    print(
        f"RAAN drift during phasing: measured {info_0['draan'] * RAD_TO_DEG:.4f} deg, "
        f"analytic {drift_analytic * RAD_TO_DEG:.4f} deg"
    )

    bias = -info_0["draan"]
    print(f"Re-running with RAAN bias {bias * RAD_TO_DEG:.4f} deg")
    tl_b, info_b = run_rpo(chaser0, TARGET_0, CFG, raan_bias=bias, verbose=True)
    use_bias = tl_b.dv_total < tl_0.dv_total
    print(
        f"J2 bias {'saves' if use_bias else 'costs'} {abs(tl_0.dv_total - tl_b.dv_total):.3f} m/s"
        f" -> using the {'biased' if use_bias else 'unbiased'} plan\n"
    )
    tl_, info_ = (tl_b, info_b) if use_bias else (tl_0, info_0)
    return {"tl": tl_, "info": info_, "raan_bias": bias if use_bias else 0.0, "tl_unbiased": tl_0}


results = {name: solve_case(name, chaser0) for name, chaser0 in CASES.items()}

print(f"{'case':<10}{'dv [m/s]':>10}{'burns':>7}{'dock [hr]':>11}{'miss [m]':>10}")
for name, r in results.items():
    t_stop = next(b.t_s for b in r["tl"].burns if b.name == "HCW stop")
    print(
        f"{name:<10}{r['tl'].dv_total:>10.2f}{len(r['tl'].burns):>7}"
        f"{t_stop / HOUR_TO_SEC:>11.2f}{norm(r['info']['docking_rsw']):>10.2f}"
    )

# Plots below use this case
PLOT_CASE = "dispersed"
tl, info = results[PLOT_CASE]["tl"], results[PLOT_CASE]["info"]
n_hat = info["n_hat"]

ts_tl, rv_c_tl, rv_t_tl = tl.arrays()

# Separation vs time: shaded phases labeled with the burns they span, plus a tick per burn
def short(name):
    """'Burn 4.3 (phasing)' -> 'Burn 4.3'."""
    return name.split(" (")[0]


rows = {r["phase"]: r for r in dv_breakdown(tl)}
bands = [
    ("plane change + Hohmann", ["Plane change", "Hohmann + phase entry"], "#e8eef9", "top left"),
    ("phasing", ["Phasing"], "#f1f1ec", "top"),
    ("phase exit + HCW", ["Phase exit", "HCW terminal"], "#fbe9e7", "top right"),
]
t_dock = rows["HCW terminal"]["t_end"]
fig_sep = plot_separation(
    ts_tl, rv_c_tl, rv_t_tl, left=0, right=(t_dock + 2 * HOUR_TO_SEC) * SEC_TO_MIN, show=False
)
for label, phases, fill, anchor in bands:
    rs = [rows[ph] for ph in phases if ph in rows]
    t0, t1 = rs[0]["t_start"], rs[-1]["t_end"]
    span = f"{short(rs[0]['first_burn'])} - {short(rs[-1]['last_burn'])}"
    fig_sep.add_vrect(
        x0=t0 * SEC_TO_MIN,
        x1=max(t1, t0 + 20 * 60) * SEC_TO_MIN,  # min width so short phases show
        fillcolor=fill,
        opacity=0.9,
        layer="below",
        line_width=0,
        annotation_text=f"<b>{span}</b><br>{label}<br>{sum(r['dv'] for r in rs):.1f} m/s",
        annotation_position=anchor,
        annotation_font={"size": 12, "color": "#52514e"},
    )

# Tick per burn along the time axis (hover for name and dv)
fig_sep.add_scatter(
    x=[b.t_s * SEC_TO_MIN for b in tl.burns],
    y=[0.0] * len(tl.burns),
    mode="markers",
    marker={"symbol": "line-ns", "size": 14, "line": {"width": 2, "color": "#52514e"}},
    text=[f"{b.name}<br>{b.dv_mag:.2f} m/s" for b in tl.burns],
    hovertemplate="%{text}<br>t = %{x:.1f} min<extra></extra>",
    cliponaxis=False,
)

# Labeled callouts for the non-phasing burns, fanned out so close burns stay readable
key = [b for b in tl.burns if "(phasing)" not in b.name]
t_mid = 0.5 * t_dock
left = [b for b in key if b.t_s < t_mid]
right = [b for b in key if b.t_s >= t_mid]
for group, sign in ((left, 1), (right, -1)):
    for k, b in enumerate(group):
        fig_sep.add_annotation(
            x=b.t_s * SEC_TO_MIN,
            y=0,
            text=f"{b.name}: {b.dv_mag:.1f} m/s",
            showarrow=True,
            arrowhead=0,
            arrowcolor="#898781",
            ax=sign * (30 + 15 * k),
            ay=-35 - 30 * k,
            xanchor="left" if sign > 0 else "right",
            font={"size": 11, "color": "#52514e"},
        )
fig_sep.update_layout(title=f"Chaser-to-target separation (total Δv {tl.dv_total:.1f} m/s)")
fig_sep.show()

===== nominal
Phasing exit (0 revs unused): sep 20.349 km (R 8.619, S -18.433, W 0.046 km), dRAAN -0.3371 deg
HCW exit (2 passes): sep 0.007 km (R -0.004, S 0.006, W -0.000 km), dRAAN -0.0000 deg
total dv = 80.566 m/s
RAAN drift during phasing: measured -0.3371 deg, analytic -0.3875 deg
Re-running with RAAN bias 0.3371 deg
Phasing exit (0 revs unused): sep 12.470 km (R 9.999, S -6.674, W -3.314 km), dRAAN -0.0330 deg
HCW exit (2 passes): sep 0.002 km (R -0.001, S 0.002, W -0.000 km), dRAAN -0.0000 deg
total dv = 83.196 m/s
J2 bias costs 2.630 m/s -> using the unbiased plan

===== dispersed
Phasing exit (0 revs unused): sep 24.009 km (R -19.664, S -8.300, W 10.993 km), dRAAN -0.2961 deg
HCW exit (2 passes): sep 0.005 km (R 0.001, S -0.002, W 0.004 km), dRAAN -0.0000 deg
total dv = 248.702 m/s
RAAN drift during phasing: measured -0.2961 deg, analytic -0.4522 deg
Re-running with RAAN bias 0.2961 deg
Phasing exit (0 revs unused): sep 20.338 km (R -19.626, S 5.312, W -0.457 km), dRAAN 0.004

### HCW Targeting

In [73]:
print(dv_breakdown_table(tl))
print()
print(tl.burn_table())
print()
for b in tl.burns:
    if b.name.startswith("HCW"):
        rr = tl.rel_at(b.t_s)
        print(f"at {b.name:<16}: |r| = {norm(rr[0:3]):9.2f} m, |v| = {norm(rr[3:6]):.4f} m/s")

phase                    burns          t [hr]   dv [m/s]   share
-----------------------------------------------------------------
Plane change                 1         1.3-1.3     129.97   52.3%
Hohmann + phase entry        2         1.4-2.4      58.39   23.5%
Phasing                     20        4.2-39.7      10.62    4.3%
Phase exit                   1       41.6-41.6      10.58    4.3%
HCW terminal                 3       41.6-42.9      39.14   15.7%
-----------------------------------------------------------------
total                       27                     248.70        

  #     t [hr]           JD (TDB)  |dv| [m/s]  name
  0      1.266     2464686.552750     129.974  Burn 1 (plane)
  1      1.420     2464686.559173      14.287  Burn 2 (Hohmann)
  2      2.387     2464686.599468      44.103  Burn 3 (circularize + phase entry)
  3      4.248     2464686.676989       0.905  Burn 4.0 (phasing)
  4      6.110     2464686.754570       0.096  Burn 4.1 (phasing)
  5      7.97

In [74]:
# Reference circles around the target
RSW_CIRCLES = [
    {"radius": 15 * KM_TO_M, "name": "HCW linear region (15 km)", "color": "#7a7a74", "dash": "dot"},
    {"radius": 200, "name": "Keep-out (200 m)", "color": "#d03b3b", "dash": "dash"},
]

ts_rsw, rv_rsw = tl.rsw_arrays(progress=True)
burns_tl = tl.burn_events()

# Full RPO, with Mars's surface for altitude context
fig_rsw = plot_rsw(
    ts_rsw[::5],
    rv_rsw[::5],
    burns_tl,
    title="RPO in the target's RSW frame",
    rho=TARGET_0.a,
    r_body=R_MARS,
    circles=RSW_CIRCLES,
)

# Terminal approach: phase exit through the HCW stop
t_exit = next(b.t_s for b in tl.burns if b.name.startswith("Burn 5"))
t_stop = next(b.t_s for b in tl.burns if b.name == "HCW stop")
m = (ts_rsw >= t_exit) & (ts_rsw <= t_stop)
fig_rsw_terminal = plot_rsw(
    ts_rsw[m],
    rv_rsw[m],
    [(t_b, name) for t_b, name in burns_tl if t_exit <= t_b <= t_stop],
    title="Terminal approach: phase exit and HCW rendezvous",
    circles=RSW_CIRCLES,
)

# Close-up: last 1 km, against the keep-out
m_close = m & (norm(rv_rsw[:, 0:3], axis=1) < 1 * KM_TO_M)
fig_rsw_close = plot_rsw(
    ts_rsw[m_close],
    rv_rsw[m_close],
    [(t_b, name) for t_b, name in burns_tl if t_b >= ts_rsw[m_close][0] and t_b <= t_stop],
    title="Final approach (inside 1 km)",
    circles=RSW_CIRCLES[1:],
)

100%|██████████| 26974/26974 [00:01<00:00, 22584.22it/s]


In [75]:
# 3D view of the chaser in the target's RSW frame (plot X/Y/Z = R/S/W), target at the origin
# Zoom on HCW: m = ts_rsw >= next(t_b for t_b, name in burns_tl if name.startswith("HCW"))
m = np.ones_like(ts_rsw, dtype=bool)
r_rel_rsw = rv_rsw[m, 0:3]
ts_rel = ts_rsw[m]

axis_len = 0.01 * norm(r_rel_rsw, axis=1).max()  # [m]
rsw_axes = [
    {
        "direction": [1, 0, 0],
        "length": axis_len,
        "name": "R",
        "color": "red",
        "label": True,
    },
    {
        "direction": [0, 1, 0],
        "length": axis_len,
        "name": "S",
        "color": "green",
        "label": True,
    },
    {
        "direction": [0, 0, 1],
        "length": axis_len,
        "name": "W",
        "color": "blue",
        "label": True,
    },
]

burn_points = [
    {"name": "Target", "location": np.zeros(3), "color": "black", "symbol": "x"}
]
for t_b, name in burns_tl:
    if t_b >= ts_rel[0] and "(phasing)" not in name:  # 20 phasing burns would flood the legend
        k = np.argmin(np.abs(ts_rel - t_b))
        burn_points.append(
            {"name": name, "location": r_rel_rsw[k], "color": "orange", "size": 4, "label": False}
        )
fig_rsw_3d = plot_3d(
    r_rel_rsw[::5],
    ts_rel[::5],
    labels="Chaser (rel. target)",
    trace_colors="royalblue",
    title="Chaser relative to target (RSW: X = R, Y = S, Z = W)",
    vectors=rsw_axes,
    points=burn_points,
    scale=KM_TO_M,  # inputs in m, displayed in km
    units="km",
)

## Visualization

In [76]:
# Node line exists only if Burn 1 fired (None for coplanar starts)
vectors = (
    []
    if n_hat is None
    else [
        {
            "direction": n_hat,
            "length": 1.6 * R_MARS * M_TO_KM,  # km, just past the surface
            "name": "Node line",
            "color": "goldenrod",
            "label": True,
        }
    ]
)

# Arrival hyperbola (t <= 0) flows into the RPO timeline (t >= 0); only its last
# stretch near Mars, so the 12 h approach doesn't shrink the orbits to a dot
near = norm(rv_approach_mci_m[:, 0:3], axis=1) < 4 * R_MARS
fig_mci = plot_3d(
    [
        rv_approach_mci_m[near, 0:3] * M_TO_KM,
        rv_t_tl[:, 0:3] * M_TO_KM,
        rv_c_tl[:, 0:3] * M_TO_KM,
    ],
    [t_approach_s[near], ts_tl, ts_tl],
    labels=["Arrival hyperbola", "Target", "Chaser"],
    trace_colors=["#b07aa1", "black", "royalblue"],
    bodies={"name": "Mars", "radius": R_MARS * M_TO_KM, "color": "orangered"},
    title=f"Arrival and RPO ({PLOT_CASE}), MCI",
    vectors=vectors,
    units="km",
)

## Save RPO
Both cases into `trajectories/rpo-<mission>.mission` for `summary.ipynb`. Times are
seconds since the capture epoch `epoch_jd`; states are MCI [m, m/s].

In [77]:
rpo_fields = {}
for name, r in results.items():
    t_, rv_c_, rv_t_ = r["tl"].arrays()
    rows_ = dv_breakdown(r["tl"])
    rpo_fields |= {
        f"{name}_time_s": t_,
        f"{name}_rv_chaser_mci_m": rv_c_,
        f"{name}_rv_target_mci_m": rv_t_,
        f"{name}_chaser0_coes": CASES[name].elements(),
        f"{name}_burn_t_s": np.array([b.t_s for b in r["tl"].burns]),
        f"{name}_burn_dv_mci_m_s": np.array([b.dv for b in r["tl"].burns]),
        f"{name}_burn_names": np.array([b.name for b in r["tl"].burns]),
        f"{name}_phase_names": np.array([row["phase"] for row in rows_]),
        f"{name}_phase_dv_m_s": np.array([row["dv"] for row in rows_]),
        f"{name}_dv_total_m_s": r["tl"].dv_total,
        f"{name}_t_dock_s": next(b.t_s for b in r["tl"].burns if b.name == "HCW stop"),
        f"{name}_miss_m": norm(r["info"]["docking_rsw"]),
        f"{name}_raan_bias_rad": r["raan_bias"],
    }
rpo_path = save_mission(
    "rpo",
    MISSION_NAME,
    epoch_jd=EPOCH0_JD_TDB,
    cases=np.array(list(results)),
    target0_coes=TARGET_0.elements(),
    j2=J2,
    seed=SEED,
    frame="MCI",
    position_units="m",
    velocity_units="m/s",
    time_origin="capture_burn",
    time_units="s",
    **rpo_fields,
)
print(rpo_path)

/Users/cskin/cesium/gnc/projects/mars-courier/trajectories/rpo-birthday_christmas_earlier.mission


## Export
Static figures and the Δv breakdown for slides.

In [78]:
FIG_DIR = Path("figures/rpo")
FIG_DIR.mkdir(parents=True, exist_ok=True)

slide_figs = {
    "separation": fig_sep,
    "rsw_full": fig_rsw,
    "rsw_terminal": fig_rsw_terminal,
    "rsw_close": fig_rsw_close,
    "rsw_3d": fig_rsw_3d,
    "mci_3d": fig_mci,
}
for name, f in slide_figs.items():
    f.write_image(FIG_DIR / f"{name}.png", width=1600, height=f.layout.height or 800, scale=2)
    print(f"wrote {FIG_DIR / name}.png")

(FIG_DIR / "dv_breakdown.txt").write_text(dv_breakdown_table(tl) + "\n")
(FIG_DIR / "burn_table.txt").write_text(tl.burn_table() + "\n")
print(f"wrote {FIG_DIR}/dv_breakdown.txt, burn_table.txt")

wrote figures/rpo/separation.png
wrote figures/rpo/rsw_full.png
wrote figures/rpo/rsw_terminal.png
wrote figures/rpo/rsw_close.png
wrote figures/rpo/rsw_3d.png
wrote figures/rpo/mci_3d.png
wrote figures/rpo/dv_breakdown.txt, burn_table.txt
